## load and prepare data

In [ ]:
%cd ../..
%matplotlib inline

import re

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.insert(0, 'evaluation_meldgraph')
from vol_eval_plots import (GROUP_3T_UNION_7T, latest_vol_eval_path, load_and_prepare_data,
                            harmo_labels, analysis_groups_clusters, harmonisation_order,
                            analysis_conditions, color_strips_by_condition,
                            label_strips_by_condition, kruskal_across_conditions,
                            describe_across_conditions)


In [ ]:
eval_stats_df = load_and_prepare_data()

## cluster stats

In [ ]:
# the per-cluster meld_graph stats in data/results/vol_eval_clusters_<timestamp>.csv
# load matching file with same timestamp
results_timestamp = re.sub(r'.*vol_eval_(.+)\.csv', r'\1', latest_vol_eval_path())
cluster_stats_df = pd.read_csv(f'data/results/vol_eval_clusters_{results_timestamp}.csv')

# same rename as for eval_stats_df above
cluster_stats_df.rename(columns={'site': 'source'}, inplace=True)

# restrict to exactly the subjects and conditions eval_stats_df has been filtered down to, and
# pick up its site / site_subj_id / analysis_group columns. the union rows carry no B0_condition
# and no clusters of their own, so they are left out
cluster_stats_df = pd.merge(
    cluster_stats_df,
    eval_stats_df.loc[eval_stats_df['analysis_group'] != GROUP_3T_UNION_7T,
                      ['model', 'harmo', 'B0_condition', 'source', 'subject ID', # matching on these
                       'site', 'site_subj_id', 'analysis_group', 'group']], # these added
    on=['model', 'harmo', 'B0_condition', 'source', 'subject ID'],
    how='inner',
    validate='many_to_one')

# the six-condition label, and the display name of its harmonisation part
cluster_stats_df['harmonisation'] = cluster_stats_df['harmo'].map(harmo_labels)
cluster_stats_df['analysis_condition'] = (cluster_stats_df['analysis_group'] + ' ' +
                                          cluster_stats_df['harmonisation'])

cluster_stats_df

In [ ]:
cluster_id_columns = ['model', 'harmo', 'harmonisation', 'B0_condition', 'source', 'subject ID',
                      'cluster', 'cluster_type', 'hemi', 'site', 'site_subj_id', 'analysis_group',
                      'analysis_condition', 'group']

cluster_stats_longdf = pd.melt(cluster_stats_df, id_vars=cluster_id_columns,
                        var_name='metric', value_name='value')


cluster_stats_longdf = cluster_stats_longdf[cluster_stats_longdf['cluster_type'] != 'unknown']

cluster_stats_longdf

In [ ]:
print('clusters per condition and type:')
print(cluster_stats_longdf
      .drop_duplicates(subset=['site_subj_id', 'analysis_condition', 'cluster', 'cluster_type'])
      .groupby(['analysis_condition', 'cluster_type']).size().unstack(fill_value=0)
      .reindex(analysis_conditions))

### Supplementary Figure 1

In [ ]:
metric_stats = cluster_stats_longdf.copy()
metric_stats = metric_stats[metric_stats['metric'].isin(['n_vertices', 'confidence'])]
metric_stats = metric_stats[metric_stats['cluster_type'].isin(['tp', 'fp'])]
grid = sns.catplot(
    data=metric_stats,
    x='analysis_group',
    order=analysis_groups_clusters,
    y='value',
    hue='harmonisation',
    hue_order=harmonisation_order,
    palette=['lightgray'] * len(harmonisation_order),
    legend=False,
    col='cluster_type',
    sharey='row',
    row='metric',
    kind='strip',
    dodge=True,
    size=3,
    height=3,
    aspect=1.2,
)
color_strips_by_condition(grid)
grid.set_xlabels('')
label_strips_by_condition(grid)

grid.fig.subplots_adjust(hspace=0.1)

# seaborn only auto-titles facets with the raw column/row values (e.g. 'fp', 'n_vertices'),
# so map those to display names by hand
col_titles = {'fp': 'False positive predictions', 'tp': 'True positive predictions'}
row_titles = {'n_vertices': 'Size [vertices]', 'confidence': 'Confidence [%]'}

letters = ['A', 'B', 'C', 'D']
for (row_val, col_val), ax in grid.axes_dict.items():
    plt.text(0.02, 0.98, letters.pop(0), transform=ax.transAxes, fontsize=12, va='top', ha='left')
    if ax.get_subplotspec().is_first_row():
        ax.set_title(col_titles[col_val])
    else:
        ax.set_title('')
    if ax.get_subplotspec().is_first_col():
        ax.set_ylabel(row_titles[row_val])

In [ ]:
# true-positive clusters, unpaired: every cluster is its own observation, so subjects with a
# true-positive cluster under only some of the six conditions still contribute
tp_cluster_stats = metric_stats[metric_stats['cluster_type'] == 'tp']

p_values_tp_df = pd.merge(describe_across_conditions(tp_cluster_stats, analysis_conditions),
                          kruskal_across_conditions(tp_cluster_stats, analysis_conditions),
                          on='metric')

p_values_tp_df

In [ ]:
# false-positive clusters, unpaired in the same way
fp_cluster_stats = metric_stats[metric_stats['cluster_type'] == 'fp']

p_values_fp_df = pd.merge(describe_across_conditions(fp_cluster_stats, analysis_conditions),
                          kruskal_across_conditions(fp_cluster_stats, analysis_conditions),
                          on='metric')

p_values_fp_df